<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자연어 처리와 LLM · 02. Bahdanau Attention

## Neural Machine Translation by Jointly Learning to Align and Translate

- **원 논문:** [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)
- **저자 / 발표:** Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio · ICLR 2015 (2014)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** additive alignment energy, softmax weights, context vector를 재현한다.

**축소하거나 생략한 부분:** bidirectional 1,000-unit GRU와 WMT 대신 24-unit GRU와 60 sentence pairs를 쓰고, 고정된 48/12 train/evaluation split을 둔다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.2.1 Eq. (5): ci=Σ αij hj | Task 1 | weights sum=1 |
| §2.2.1 Eq. (6): alignment softmax | Task 1 | probability assert |
| §2.2.1 Eq. (7): a(si-1,hj) | Task 1 AdditiveAttention | energy shape |
| §3.1 Eq. (8–9): bidirectional annotations | Task 2–3 | attention heatmap |

## 핵심 아이디어와 수식

$$e_{ij}=v_a^{\top}\tanh(W_as_{i-1}+U_ah_j),\quad \alpha_{ij}=\frac{\exp e_{ij}}{\sum_k\exp e_{ik}},\quad c_i=\sum_j\alpha_{ij}h_j$$

**기호 해설:** s는 decoder state, h는 encoder annotation, e는 energy, α는 alignment, c는 context다.

**코드 대응:** Task 1의 AdditiveAttention이 세 식, Task 2가 step decoder, Task 3이 CE/heatmap을 맡는다.

**입력과 출력 shape:** encoder [B,3,24], query [B,24] → weights [B,3] → context [B,24].

**포트폴리오 구현 범위:** 양방향 encoder 대신 단방향 GRU를 쓰되 동적 context와 alignment normalization은 같다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2.1 Eq. (5): ci=Σ αij hj
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** weights sum=1

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
@dataclass
class AttentionConfig:
    embedding_dim: int = 16
    hidden_dim: int = 24
    learning_rate: float = 0.025
    steps: int = 26


class AdditiveAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.query_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.key_projection = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.energy_projection = nn.Linear(hidden_dim, 1, bias=False)

    def energies(self, query, annotations):
        projected_query = self.query_projection(query)[:, None]
        projected_keys = self.key_projection(annotations)
        return self.energy_projection(
            torch.tanh(projected_query + projected_keys)
        ).squeeze(-1)

    def forward(self, query, annotations, valid_mask=None):
        energy = self.energies(query, annotations)
        if valid_mask is not None:
            energy = energy.masked_fill(~valid_mask, float("-inf"))
        weights = energy.softmax(dim=-1)
        context = torch.bmm(weights[:, None], annotations).squeeze(1)
        return context, weights


config = AttentionConfig()
attention = AdditiveAttention(config.hidden_dim).to(DEVICE)
query_probe = torch.randn(2, 24, device=DEVICE)
annotation_probe = torch.randn(2, 3, 24, device=DEVICE)
context_probe, weight_probe = attention(query_probe, annotation_probe)
assert context_probe.shape == (2, 24)
assert torch.allclose(weight_probe.sum(dim=1), weight_probe.new_ones(2))

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §2.2.1 Eq. (6): alignment softmax
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** probability assert

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
class AttentiveSeq2Seq(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config.embedding_dim)
        self.encoder = nn.GRU(
            config.embedding_dim,
            config.hidden_dim,
            batch_first=True,
        )
        self.attention = AdditiveAttention(config.hidden_dim)
        decoder_input_dim = config.embedding_dim + config.hidden_dim
        self.decoder_cell = nn.GRUCell(decoder_input_dim, config.hidden_dim)
        self.output = nn.Linear(config.hidden_dim * 2, vocab_size)

    def encode(self, source):
        annotations, final_state = self.encoder(self.embedding(source))
        return annotations, final_state.squeeze(0)

    def decode_step(self, token, state, annotations, valid_mask):
        context, weights = self.attention(state, annotations, valid_mask)
        embedded = self.embedding(token)
        state = self.decoder_cell(torch.cat((embedded, context), dim=-1), state)
        logits = self.output(torch.cat((state, context), dim=-1))
        return logits, state, weights

    def forward(self, source, decoder_inputs):
        annotations, state = self.encode(source)
        valid_mask = source != PAD
        logits_by_step = []
        weights_by_step = []
        for step in range(decoder_inputs.shape[1]):
            logits, state, weights = self.decode_step(
                decoder_inputs[:, step],
                state,
                annotations,
                valid_mask,
            )
            logits_by_step.append(logits)
            weights_by_step.append(weights)
        return torch.stack(logits_by_step, 1), torch.stack(weights_by_step, 1)


split_generator = torch.Generator().manual_seed(22)
translation_order = torch.randperm(
    len(source_ids),
    generator=split_generator,
).to(source_ids.device)
train_translation_index = translation_order[:48]
test_translation_index = translation_order[48:]
assert not bool(
    torch.isin(
        train_translation_index,
        test_translation_index,
    ).any()
)
train_source = source_ids[train_translation_index]
test_source = source_ids[test_translation_index]
train_targets = target_ids[train_translation_index]
test_targets = target_ids[test_translation_index]
train_decoder_inputs = torch.cat(
    (train_targets.new_full((48, 1), BOS), train_targets[:, :-1]),
    dim=1,
)
test_decoder_inputs = torch.cat(
    (test_targets.new_full((12, 1), BOS), test_targets[:, :-1]),
    dim=1,
)
model = AttentiveSeq2Seq(VOCAB_SIZE, config).to(DEVICE)
logits, alignment = model(train_source, train_decoder_inputs)
assert logits.shape == (48, 3, VOCAB_SIZE)
assert alignment.shape == (48, 3, 3)

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Eq. (8–9): bidirectional annotations
- **노트북에서 구현할 부분:** Task 2–3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** attention heatmap

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
def attentive_loss(logits, targets):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=PAD,
    )


def train_attention_step(model, optimizer, source, decoder_inputs, targets):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    logits, _ = model(source, decoder_inputs)
    loss = attentive_loss(logits, targets)
    loss.backward()
    optimizer.step()
    return float(loss.detach())


def evaluate_attention(model, source, decoder_inputs, targets):
    model.eval()
    with torch.no_grad():
        logits, weights = model(source, decoder_inputs)
    valid = targets != PAD
    accuracy = (logits.argmax(-1)[valid] == targets[valid]).float().mean()
    return float(accuracy), weights


optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
losses = []
for step in range(config.steps):
    losses.append(
        train_attention_step(
            model,
            optimizer,
            train_source,
            train_decoder_inputs,
            train_targets,
        )
    )
heldout_accuracy, heldout_weights = evaluate_attention(
    model,
    test_source,
    test_decoder_inputs,
    test_targets,
)
assert min(losses[3:]) < losses[0]
assert torch.allclose(
    heldout_weights.sum(dim=-1),
    heldout_weights.new_ones(12, 3),
)
plt.figure(figsize=(4, 3))
plt.imshow(heldout_weights[0].detach().cpu())
plt.xlabel("source position")
plt.ylabel("target position")
plt.title(f"held-out alignment, token acc={heldout_accuracy:.1%}")
plt.colorbar()
plt.tight_layout()
plt.show()

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> bidirectional 1,000-unit GRU와 WMT 대신 24-unit GRU와 60 sentence pairs를 쓰고, 고정된 48/12 train/evaluation split을 둔다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.